In [18]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

In [19]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

In [20]:
url = "https://www.tealhq.com/resume-examples"
driver.get(url)

In [ ]:
wait = WebDriverWait(driver, 20)
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

<selenium.webdriver.remote.webelement.WebElement (session="d68bf3aba2826bb3e40eeb748b2af69b", element="f.D643E989204183A82444BF4A750D796A.d.2629D14A9500FB05085B031A40EB8939.e.20")>

In [23]:
def scroll_to_bottom():
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

In [24]:
scroll_to_bottom()
time.sleep(1)

In [ ]:
sections = driver.find_elements(By.CSS_SELECTOR, "section.u--section")

In [26]:
print(f"Trang có {len(sections)} thẻ <section.u--container>")

Trang có 10 thẻ <section.u--container>


In [ ]:
if len(sections) < 3:
    raise Exception("Không tìm thấy đủ section, kiểm tra lại cấu trúc trang!")

In [ ]:
target_section = sections[2]

In [ ]:
category_ids = [
    "Data-Analytics",
    "Design-UX",
    "Dev-Engineering",
    "Information-Technology"
]

In [30]:
data = []

In [ ]:
for cat_id in category_ids:
    try:
        category_div = target_section.find_element(By.ID, cat_id)

        js = """
        const root = arguments[0];
        const anchors = root.querySelectorAll('a[aria-label="Link to resume example"]');
        const results = [];
        anchors.forEach(a => {
            const div = a.querySelector('div');
            const text = div ? div.innerText.trim() : '';
            results.push({href: a.getAttribute('href'), category: text});
        });
        return results;
        """
        items = driver.execute_script(js, category_div)

        if not items:
            driver.execute_script("""
            const hidden = arguments[0].querySelectorAll('.w-dyn-item.hide');
            hidden.forEach(el => {
                el.classList.remove('hide');
                el.style.display = 'block';
                el.style.visibility = 'visible';
            });
            """, category_div)
            time.sleep(0.5)
            items = driver.execute_script(js, category_div)

        for item in items:
            href = item["href"]
            if href.startswith("/"):
                href = "https://www.tealhq.com" + href
            data.append({
                "Category": item["category"],
                "Link": href
            })
        print(f"✅ {cat_id}: lấy được {len(items)} items")
    except Exception as e:
        print(f"⚠️ Lỗi khi xử lý {cat_id}: {e}")

✅ Data-Analytics: lấy được 31 items
✅ Design-UX: lấy được 27 items
✅ Dev-Engineering: lấy được 85 items
✅ Information-Technology: lấy được 31 items


In [35]:
print(data)

[{'Category': 'Director of Analytics', 'Link': 'https://www.tealhq.com/resume-examples/director-of-analytics', 'GroupID': 'Data-Analytics'}, {'Category': 'GIS', 'Link': 'https://www.tealhq.com/resume-examples/gis', 'GroupID': 'Data-Analytics'}, {'Category': 'Neuroscientist', 'Link': 'https://www.tealhq.com/resume-examples/neuroscientist', 'GroupID': 'Data-Analytics'}, {'Category': 'Performance Analyst', 'Link': 'https://www.tealhq.com/resume-examples/performance-analyst', 'GroupID': 'Data-Analytics'}, {'Category': 'Quant', 'Link': 'https://www.tealhq.com/resume-examples/quant', 'GroupID': 'Data-Analytics'}, {'Category': 'Statistician', 'Link': 'https://www.tealhq.com/resume-examples/statistician', 'GroupID': 'Data-Analytics'}, {'Category': 'Data Analyst', 'Link': 'https://www.tealhq.com/resume-examples/data-analyst', 'GroupID': 'Data-Analytics'}, {'Category': 'Data Engineer', 'Link': 'https://www.tealhq.com/resume-examples/data-engineer', 'GroupID': 'Data-Analytics'}, {'Category': 'Bus

In [36]:
driver.quit()

In [37]:
if data:
    df = pd.DataFrame(data)
    output_path = "D:/BaiDoAnChuyenNganh3/Automated-Resume-Ranking-System-main/csvfiles/crawlcv/tealhq_links.csv"
    df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"\n✅ Đã lưu {len(data)} mẫu resume vào file:\n{output_path}")
else:
    print("⚠️ Không có dữ liệu nào được lấy!")


✅ Đã lưu 348 mẫu resume vào file:
D:/BaiDoAnChuyenNganh3/Automated-Resume-Ranking-System-main/csvfiles/crawlcv/tealhq_links.csv
